# Electricity access and global D1

This notebook compares the country share of the global bottom decile with electricity coverage. The deprivation proxy is `100 - access percentage`.

In [27]:
from pathlib import Path
import sys
import importlib
import altair as alt

NOTEBOOK_DIR = Path.cwd() if (Path.cwd() / 'indicator_analysis_helpers.py').exists() else Path.cwd() / 'dev_indicators'
sys.path.insert(0, str(NOTEBOOK_DIR))
import indicator_analysis_helpers
importlib.reload(indicator_analysis_helpers)
from indicator_analysis_helpers import REGION_COLORS, run_indicator_analysis

result = run_indicator_analysis('WB_WDI', 'WB_WDI_EG_ELC_ACCS_ZS', 'coverage_pct')
plot_df = result['joined'][result['joined']['year'].isin([result['first_year'], result['last_year']])].copy()
plot_df = plot_df.dropna(subset=['region'])
plot_df['year'] = plot_df['year'].astype(str)
plot_years = [str(result['first_year']), str(result['last_year'])]

In [28]:
print(f"Common years: {result['first_year']} and {result['last_year']}")
print(result['deprivation_label'])
display(result['correlations'].round(3))
display(result['endpoint_summary'].round(3))
result['chart']

Common years: 1990 and 2024
Estimated population without access (%) = 100 - coverage


,year,pearson_r,spearman_r,observations
0,1990,0.828,0.492,98
1,1991,0.767,0.622,109
2,1992,0.792,0.667,123
3,1993,0.779,0.716,132
4,1994,0.769,0.705,138
5,1995,0.773,0.758,145
6,1996,0.750,0.777,154
7,1997,0.769,0.773,160
8,1998,0.781,0.786,166
9,1999,0.765,0.781,172


,year,countries,mean_indicator,mean_deprivation_proxy,median_deprivation_proxy,observed_population,population_weighted_indicator,population_weighted_deprivation_proxy
0,1990,98,96.298,3.702,0.0,1587.393,92.484,7.516
1,2024,215,88.786,11.214,0.0,8116.755,91.922,8.078


alt.Chart(...)

## Scatter plot: electricity access and global D1

Each bubble is a country. The horizontal axis is the percentage of that country's population in the global D1, the vertical axis is national electricity access, and bubble size reflects the population weight. The two panels show the first and last common years.

In [29]:
point_chart = (
    alt.Chart(plot_df)
    .mark_circle(opacity=0.7, stroke='white', strokeWidth=0.5)
    .encode(
        x=alt.X('d1_country_population_pct:Q', title='Population in global D1 (%)'),
        y=alt.Y('indicator_value:Q', title='Population with access to electricity (%)', scale=alt.Scale(domain=[0, 100])),
        color=alt.Color('region:N', title='World Bank region', scale=alt.Scale(domain=list(REGION_COLORS), range=list(REGION_COLORS.values()))),
        size=alt.Size('country_population:Q', legend=None, scale=alt.Scale(range=[40, 900])),
        tooltip=[
            alt.Tooltip('code:N', title='Country'),
            alt.Tooltip('region:N', title='Region'),
            alt.Tooltip('year:O', title='Year'),
            alt.Tooltip('d1_country_population_pct:Q', title='Population in D1 (%)', format='.1f'),
            alt.Tooltip('indicator_value:Q', title='Electricity access (%)', format='.1f'),
        ],
    )
    .properties(width=420, height=420)
)
electricity_scatter = point_chart.facet(column=alt.Column('year:O', title=None, sort=plot_years)).properties(title='Electricity access versus population in the global D1')
electricity_scatter

alt.FacetChart(...)

## Where the poorest decile lives

This measure only counts **where people live**, so it makes no assumption about who within a country has access.

For each year it reports the share of the global poorest decile living in countries where fewer than half the population has electricity, next to the same share for the world population. The ratio between the two lines is the gap.

In [30]:
from indicator_analysis_helpers import low_coverage_chart

low_coverage = result['low_coverage']
display(low_coverage.iloc[[0, -1]].round(2))
print(f"Gap ratio: {low_coverage.iloc[0]['gap_ratio']:.2f}x in {low_coverage.iloc[0]['year']:.0f}"
      f" -> {low_coverage.iloc[-1]['gap_ratio']:.2f}x in {low_coverage.iloc[-1]['year']:.0f}")
low_coverage_chart(low_coverage, 'Electricity access')

,year,d1_share_below,world_share_below,gap_ratio,countries_below,countries_with_data
0,1991,74.69,23.65,3.16,8,109
33,2024,28.42,4.33,6.57,15,215


Gap ratio: 3.16x in 1991 -> 6.57x in 2024


alt.Chart(...)